# Triplet Network

## Import Libraries

In [ ]:
!pip install datasets -q

import os
import datasets
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
import torch
from PIL import Image
import torchvision
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
import os
from pathlib import Path
import torch


class Config:
    """
    Configuration class for the project.
    """
    # ========== Base Paths ==========
    project_name = "logo_recognition_similarity_search_project"
    base_dir = Path("/content/drive/MyDrive")/project_name
    output_dir = base_dir / "output"
    cache_dir = base_dir / "cache"

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(cache_dir, exist_ok=True)

    # ========== Dataset ==========
    dataset_path = "mlproject5606/Logo-Recognition-Category-ResNet50-Backbone-Embeddings-Classess-Dataset"
    dataset_cache_dir = cache_dir / dataset_path.split("/")[-1]
    category_indices_cache_dir = cache_dir / "Category-Indices" / dataset_path.split("/")[-1]
    batch_size = 2000


    # ========== Runtime ==========
    device = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers = os.cpu_count()
    seed = 42


# Create Configuration Object
cfg = Config()

## Load Dataset

In [ ]:
# from datasets import load_dataset

# ds = load_dataset(path = cfg.dataset_path,
#                   split = "train",
#                   )
# ds

In [ ]:
from datasets import load_from_disk

ds = load_from_disk(cfg.dataset_cache_dir)
ds

Loading dataset from disk:   0%|          | 0/164 [00:00<?, ?it/s]

Dataset({
    features: ['id', 'path', 'path_category', 'prompt', 'image', 'category', 'resnet50_embedding', 'resnet50_class'],
    num_rows: 1777584
})

In [ ]:
# Inspect the structure of the dataset

ds.features

{'id': Value(dtype='string', id=None),
 'path': Value(dtype='string', id=None),
 'path_category': Value(dtype='string', id=None),
 'prompt': Value(dtype='string', id=None),
 'image': Image(mode='RGB', decode=True, id=None),
 'category': Value(dtype='int32', id=None),
 'resnet50_embedding': Sequence(feature=Sequence(feature=Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None), length=-1, id=None), length=-1, id=None),
 'resnet50_class': [{'label': Value(dtype='string', id=None),
   'score': Value(dtype='float64', id=None)}]}

In [ ]:
# Inspect the verify the output of resnet50_embedding and resnet50_class columns

(
 len(ds[0]['resnet50_embedding']),
 len(ds[0]['resnet50_class']),
 ds[0]['resnet50_class']
)

(2048,
 5,
 [{'label': 'envelope', 'score': 0.0832},
  {'label': 'jersey', 'score': 0.06026},
  {'label': 'binder', 'score': 0.02756},
  {'label': 'nipple', 'score': 0.02047},
  {'label': 'ballpoint', 'score': 0.01766}])

## Triplet Dataset

In [ ]:
import os
import json
from collections import defaultdict
from tqdm import tqdm

def load_or_create_category_indices(ds, file_path):
    """
    Load category_to_indices mapping from file if exists,
    otherwise create it from the dataset and save as JSON.

    Args:
        ds (datasets.Dataset): The dataset containing 'category' field.
        file_path (str): Path to the JSON file.

    Returns:
        dict: category_to_indices mapping.
    """
    if os.path.exists(file_path):
        print(f"[INFO] Loading category_to_indices from: {file_path}")
        with open(file_path, 'r') as f:
            category_to_indices = json.load(f)
        # Convert string keys back to int (JSON saves keys as strings)
        category_to_indices = {int(k): v for k, v in category_to_indices.items()}
    else:
        print(f"[INFO] Creating category_to_indices from dataset...")
        category_to_indices = defaultdict(list)

        for idx in tqdm(range(len(ds)), desc="Building category index"):
            row = ds[idx]
            category = row["category"]
            category_to_indices[category].append(idx)

        category_to_indices = dict(category_to_indices)

        with open(file_path, 'w') as f:
            json.dump(category_to_indices, f)

        print(f"[INFO] Saved category_to_indices to: {file_path}")

    return category_to_indices


In [ ]:
category_index_path = cfg.category_indices_cache_dir / "category_to_indices.json"
category_to_indices = load_or_create_category_indices(ds, str(category_index_path))
len(category_to_indices)

[INFO] Creating category_to_indices from dataset...


Building category index:  24%|██▍       | 434979/1777584 [1:59:26<6:58:05, 53.52it/s]

In [ ]:
def split_dataset_by_category_indices(ds, category_to_indices, test_ratio=0.1, seed=42):
    """
    Split dataset into train/test indices by category.
    """
    import random
    random.seed(seed)

    train_indices = []
    test_indices = []

    for cat, indices in tqdm(category_to_indices.items(), desc="Splitting train/test"):
        if len(indices) < 2:
            continue

        indices_copy = indices.copy()
        random.shuffle(indices_copy)

        test_count = max(1, int(len(indices_copy) * test_ratio))

        if len(indices_copy)-test_count < 2:
            train = indices_copy[:]
            test = []
            print(f"Skipping category {cat} due to insufficient images in test set")
        else:
            test = indices_copy[:test_count]
            train = indices_copy[test_count:]


        train_indices.extend(train)
        test_indices.extend(test)

    ds_train = ds.select(train_indices)
    ds_test = ds.select(test_indices)

    return ds_train, ds_test


In [ ]:
ds_train, ds_test = split_dataset_by_category_indices(ds,
                                                      category_to_indices = category_to_indices,
                                                      test_ratio=0.2,
                                                      seed=cfg.seed
                                                      )

In [ ]:
from datasets import DatasetDict

split_dataset = DatasetDict({
    "train": ds_train,
    "test": ds_test
})

repo_id = "mlproject5606/Logo-Recognition-Category-ResNet50-Backbone-Embeddings-Classess-Splits-Dataset"
split_dataset.push_to_hub(repo_id=repo_id, private=True)



---



In [ ]:
import os
import json
import torch
from torch.utils.data import Dataset
from collections import defaultdict
from tqdm import tqdm
import random

class TripletLogoDataset(Dataset):
    def __init__(self,ds: datasets.Dataset, category_indices_path:str ):
        """
        Initialize the TripletLogoDataset.
        """
        self.ds = ds
        self.rng = random.Random()
        self.category_indices_path = category_indices_path
        self.category_to_indices = self._load_or_create_category_indices()
        self.categories = list(self.category_to_indices.keys())

    def _load_or_create_category_indices(self):
        """
        Load category_to_indices from disk if available,
        otherwise create and save it.
        """
        if os.path.exists(self.category_indices_path):
            print(f"[INFO] Loading category_to_indices from {self.category_indices_path}")
            with open(self.category_indices_path, "r") as f:
                data = json.load(f)
            return {int(k): v for k, v in data.items()}
        else:
            print(f"[INFO] Creating category_to_indices ...")
            category_to_indices = defaultdict(list)
            for idx in tqdm(range(len(self.ds)), desc="Building category index"):
                category = self.ds[idx]["category"]
                category_to_indices[category].append(idx)

            # Save to disk
            with open(self.category_indices_path, "w") as f:
                json.dump(category_to_indices, f)

            print(f"[INFO] Saved category_to_indices to {self.category_indices_path}")
            return dict(category_to_indices)

    def __len__(self):
        """
        Return the number of samples in the dataset.
        """
        return len(self.ds)

    def _get_positive_index(self, anchor_idx, category):
        """
        Get a positive index for the given anchor index and category.
        """
        positive_indices = self.category_to_indices[category].copy()
        positive_indices.remove(anchor_idx)
        if not positive_indices:
            print(f"No positive indices found for category {category}")
            return anchor_idx
        positive_idx = self.rng.choice(positive_indices)
        return positive_idx

    def _get_negative_index(self, anchor_idx, category):
        """
        Get a negative index for the given anchor index and category.
        """
        negative_categories = self.categories.copy()
        negative_categories.remove(category)
        negative_category = self.rng.choice(negative_categories)
        negative_indices = self.category_to_indices[negative_category]
        negative_idx = self.rng.choice(negative_indices)
        return negative_idx

    def __getitem__(self, idx):
        """
        Get a triplet sample from the dataset.
        """
        anchor_idx = idx
        anchor = self.ds[anchor_idx]
        category = anchor["category"]

        positive_idx = self._get_positive_index(anchor_idx, category)
        positive = self.ds[positive_idx]

        negative_idx = self._get_negative_index(anchor_idx, category)
        negative = self.ds[negative_idx]

        return anchor, positive, negative

    @staticmethod
    def collate_fn(batch):
        """
        Collate function for the DataLoader.
        """
        anchors, positives, negatives = zip(*batch)
        anchors_embedding = torch.stack([torch.tensor(a["resnet50_embedding"]) for a in anchors])
        positives_embedding = torch.stack([torch.tensor(p["resnet50_embedding"]) for p in positives])
        negatives_embedding = torch.stack([torch.tensor(n["resnet50_embedding"]) for n in negatives])
        return anchors_embedding, positives_embedding, negatives_embedding




In [ ]:
train_category_indices_path = cfg.category_indices_cache_dir / "category_train_indices.json"

train_dataset = TripletLogoDataset(ds = ds_train,
                                  category_indices_path=train_category_indices_path
                                  )

In [ ]:
train_loader  = torch.utils.data.DataLoader(train_dataset,
                                            batch_size=cfg.batch_size,
                                            collate_fn=TripletLogoDataset.collate_fn
                                            )

batch = next(iter(train_loader))
print(batch[0].shape)

## Triplet Dataset2

In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms
import datasets
import random
from collections import defaultdict
from tqdm import tqdm
import pickle
import os

class TripletLogoDataset(torch.utils.data.Dataset):


    def __init__(self,ds: datasets.Dataset, cfg: Config, dataset_type: str = "train"):
        """
        Initialize the TripletLogoDataset.
        """

        if dataset_type not in ["train", "validation", "test"]:
            raise ValueError(f"Invalid dataset type: {dataset_type}")
        self.dataset_type = dataset_type
        self.ds = ds[dataset_type]
        self.cfg = cfg


        os.makedirs(cfg.category_indices_cache_dir, exist_ok=True)

        if dataset_type == "train":
             self.rng = random.Random()
             category_path = os.path.join(cfg.category_indices_cache_dir, 'category_train_indices.pkl')
        elif dataset_type == "validation":
             self.rng = random.Random(cfg.seed)
             category_path = os.path.join(cfg.category_indices_cache_dir, 'category_validation_indices.pkl')
        else:
             self.rng = random.Random(cfg.seed)
             category_path = os.path.join(cfg.category_indices_cache_dir, 'category_test_indices.pkl')

        if os.path.exists(category_path):
            with open(category_path, 'rb') as f:
                self.category_to_indices = pickle.load(f)
        else:
            self.category_to_indices = self._build_category_indices()
            with open(category_path, 'wb') as f:
                pickle.dump(self.category_to_indices, f)


        self.categories = list(self.category_to_indices.keys())


    def _build_category_indices(self):
        """
        Build a dictionary of category to indices.
        """
        category_to_indices = defaultdict(list)
        for idx, example in enumerate(tqdm(self.ds, desc="Building category to indices")):
            category_to_indices[example['category']].append(idx)
        return category_to_indices

    def __len__(self):
        """
        Return the number of samples in the dataset.
        """
        return len(self.ds)

    def _get_positive_index(self, anchor_idx, category):
        """
        Get a positive index for the given anchor index and category.
        """
        positive_indices = self.category_to_indices[category][:]
        positive_indices.remove(anchor_idx)
        positive_idx = self.rng.choice(positive_indices)
        return positive_idx

    def _get_negative_index(self, anchor_idx, category):
        """
        Get a negative index for the given anchor index and category.
        """
        negative_categorys = self.categories[:]
        negative_categorys.remove(category)
        negative_category = self.rng.choice(negative_categorys)
        negative_indices = self.category_to_indices[negative_category]
        negative_idx = self.rng.choice(negative_indices)
        return negative_idx

    def __getitem__(self, idx):
        """
        Get a triplet sample from the dataset.
        """
        anchor_idx = idx
        anchor = self.ds[anchor_idx]
        category = anchor['category']

        positive_idx = self._get_positive_index(anchor_idx, category)
        positive = self.ds[positive_idx]

        negative_idx = self._get_negative_index(anchor_idx, category)
        negative = self.ds[negative_idx]

        return anchor , positive , negative


In [ ]:
triplet_train_ds = TripletLogoDataset(ds=ds, cfg=cfg, dataset_type="train")
triplet_val_ds = TripletLogoDataset(ds=ds, cfg=cfg, dataset_type="validation")
triplet_test_ds = TripletLogoDataset(ds=ds, cfg=cfg, dataset_type="test")

In [ ]:
anchor , positive , negative = triplet_train_ds[4]
display(anchor['image'],positive['image'],negative['image'])

## Networks

In [ ]:
import torch.nn as nn
from torchvision import models

class Backbone(nn.Module):
    """
    ResNet50 backbone for triplet network
    """
    def __init__(self,freeze: bool = True):
        """
        Initialize the backbone
        """
        super(Backbone, self).__init__()
        self.model = models.resnet50(pretrained=True)
        # Remove the last fully connected layer
        self.out_features = self.model.fc.in_features
        self.model = torch.nn.Sequential(*list(self.model.children())[:-1])
        # Freeze layers
        if freeze:
            for param in self.model.parameters():
                param.requires_grad = False


    def forward(self, x):
        """
        Forward pass of the backbone
        """
        return self.model(x)

# .............................................................................

class EmbeddingNet(nn.Module):
    """
    Embedding network for triplet network
    """
    def __init__(self, backbone, embedding_size=256):
        super(EmbeddingNet, self).__init__()
        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(self.backbone.out_features, 512),
            nn.ReLU(),
            nn.Linear(512, embedding_size)
        )

    def forward(self, x):
        """
        Forward pass of the embedding network
        """
        features = self.backbone(x)
        features = features.flatten(1)
        embedding = self.embedding(features)
        embedding = nn.functional.normalize(embedding, p=2, dim=1)
        return embedding

    def get_embedding(self, x):
        """
        Get the embedding of an input image
        """
        return self.forward(x)

# .............................................................................

class TripletNet(nn.Module):
    """
    Triplet network for logo recognition
    """
    def __init__(self, embedding_net):
        super(TripletNet, self).__init__()
        self.embedding_net = embedding_net

    def forward(self, x1, x2, x3):
        """
        Forward pass of the triplet network
        """
        output1 = self.embedding_net(x1)
        output2 = self.embedding_net(x2)
        output3 = self.embedding_net(x3)
        return output1, output2, output3

    def get_embedding(self, x):
        """
        Get the embedding of an input image
        """
        return self.embedding_net(x)


In [ ]:
backbone = Backbone()
embedding_net = EmbeddingNet(backbone)
triplet_net = TripletNet(embedding_net)

In [ ]:
anchor_image_tensor = cfg.transform(anchor['image']).unsqueeze(0)
positive_image_tensor = cfg.transform(positive['image']).unsqueeze(0)
negative_image_tensor = cfg.transform(negative['image']).unsqueeze(0)

In [ ]:
backbone(anchor_image_tensor).shape,embedding_net(anchor_image_tensor).shape

In [ ]:
anchor_output,positive_output,negative_output = triplet_net(anchor_image_tensor,positive_image_tensor,negative_image_tensor)
anchor_output.shape,positive_output.shape,negative_output.shape

## Losses

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TripletLoss(nn.Module):
    """
    Triplet loss
    Takes embeddings of an anchor sample, a positive sample and a negative sample
    """

    def __init__(self, margin):
        super(TripletLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative, size_average=True):
        distance_positive = (anchor - positive).pow(2).sum(1)  # .pow(.5)
        distance_negative = (anchor - negative).pow(2).sum(1)  # .pow(.5)
        losses = F.relu(distance_positive - distance_negative + self.margin)
        return losses.mean() if size_average else losses.sum()



## Metrics

In [ ]:
import numpy as np


class Metric:
    def __init__(self):
        pass

    def __call__(self, outputs, target, loss):
        raise NotImplementedError

    def reset(self):
        raise NotImplementedError

    def value(self):
        raise NotImplementedError

    def name(self):
        raise NotImplementedError


class AccumulatedAccuracyMetric(Metric):
    """
    Works with classification model
    """

    def __init__(self):
        self.correct = 0
        self.total = 0

    def __call__(self, outputs, target, loss):
        pred = outputs[0].data.max(1, keepdim=True)[1]
        self.correct += pred.eq(target[0].data.view_as(pred)).cpu().sum()
        self.total += target[0].size(0)
        return self.value()

    def reset(self):
        self.correct = 0
        self.total = 0

    def value(self):
        return 100 * float(self.correct) / self.total

    def name(self):
        return 'Accuracy'


class AverageNonzeroTripletsMetric(Metric):
    '''
    Counts average number of nonzero triplets found in minibatches
    '''

    def __init__(self):
        self.values = []

    def __call__(self, outputs, target, loss):
        self.values.append(loss[1])
        return self.value()

    def reset(self):
        self.values = []

    def value(self):
        return np.mean(self.values)

    def name(self):
        return 'Average nonzero triplets'


## Trainer

In [ ]:
import torch
import numpy as np


def fit(train_loader, val_loader, model, loss_fn, optimizer, scheduler, n_epochs, cuda, log_interval, metrics=[],
        start_epoch=0):
    """
    Loaders, model, loss function and metrics should work together for a given task,
    i.e. The model should be able to process data output of loaders,
    loss function should process target output of loaders and outputs from the model

    Examples: Classification: batch loader, classification model, NLL loss, accuracy metric
    Siamese network: Siamese loader, siamese model, contrastive loss
    Online triplet learning: batch loader, embedding model, online triplet loss
    """
    for epoch in range(0, start_epoch):
        scheduler.step()

    for epoch in range(start_epoch, n_epochs):
        scheduler.step()

        # Train stage
        train_loss, metrics = train_epoch(train_loader, model, loss_fn, optimizer, cuda, log_interval, metrics)

        message = 'Epoch: {}/{}. Train set: Average loss: {:.4f}'.format(epoch + 1, n_epochs, train_loss)
        for metric in metrics:
            message += '\t{}: {}'.format(metric.name(), metric.value())

        val_loss, metrics = test_epoch(val_loader, model, loss_fn, cuda, metrics)
        val_loss /= len(val_loader)

        message += '\nEpoch: {}/{}. Validation set: Average loss: {:.4f}'.format(epoch + 1, n_epochs,
                                                                                 val_loss)
        for metric in metrics:
            message += '\t{}: {}'.format(metric.name(), metric.value())

        print(message)


def train_epoch(train_loader, model, loss_fn, optimizer, cuda, log_interval, metrics):
    for metric in metrics:
        metric.reset()

    model.train()
    losses = []
    total_loss = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        target = target if len(target) > 0 else None
        if not type(data) in (tuple, list):
            data = (data,)
        if cuda:
            data = tuple(d.cuda() for d in data)
            if target is not None:
                target = target.cuda()


        optimizer.zero_grad()
        outputs = model(*data)

        if type(outputs) not in (tuple, list):
            outputs = (outputs,)

        loss_inputs = outputs
        if target is not None:
            target = (target,)
            loss_inputs += target

        loss_outputs = loss_fn(*loss_inputs)
        loss = loss_outputs[0] if type(loss_outputs) in (tuple, list) else loss_outputs
        losses.append(loss.item())
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

        for metric in metrics:
            metric(outputs, target, loss_outputs)

        if batch_idx % log_interval == 0:
            message = 'Train: [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                batch_idx * len(data[0]), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), np.mean(losses))
            for metric in metrics:
                message += '\t{}: {}'.format(metric.name(), metric.value())

            print(message)
            losses = []

    total_loss /= (batch_idx + 1)
    return total_loss, metrics


def test_epoch(val_loader, model, loss_fn, cuda, metrics):
    with torch.no_grad():
        for metric in metrics:
            metric.reset()
        model.eval()
        val_loss = 0
        for batch_idx, (data, target) in enumerate(val_loader):
            target = target if len(target) > 0 else None
            if not type(data) in (tuple, list):
                data = (data,)
            if cuda:
                data = tuple(d.cuda() for d in data)
                if target is not None:
                    target = target.cuda()

            outputs = model(*data)

            if type(outputs) not in (tuple, list):
                outputs = (outputs,)
            loss_inputs = outputs
            if target is not None:
                target = (target,)
                loss_inputs += target

            loss_outputs = loss_fn(*loss_inputs)
            loss = loss_outputs[0] if type(loss_outputs) in (tuple, list) else loss_outputs
            val_loss += loss.item()

            for metric in metrics:
                metric(outputs, target, loss_outputs)

    return val_loss, metrics
